<!-- derived-from: ../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/PDF%E6%95%B0%E6%8D%AE%E6%B8%85%E6%B4%97%E8%84%9A%E6%9C%AC%E7%A4%BA%E4%BE%8B.ipynb -->
> English mirror, derived from the Chinese master: [the PDF cleaning script example](../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/PDF%E6%95%B0%E6%8D%AE%E6%B8%85%E6%B4%97%E8%84%9A%E6%9C%AC%E7%A4%BA%E4%BE%8B.ipynb). The Chinese notebook is the source of truth.
>
> Outputs are intentionally not stored here: this notebook writes into `../../knowledge_md/VMAX-S/`, so run it yourself rather than trusting a transcript.
## Built on the unstructured library


In [ ]:
#!/usr/bin/env python
# coding: utf-8


from pathlib import Path
import re


from unstructured.partition.pdf import partition_pdf

from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Table,
    PageBreak,
    Header,
    Footer,
)



# =====================================================
# path configuration
# =====================================================

INPUT_DIR = Path(
    "../knowledge_path/VMAX-S"
)


OUTPUT_DIR = Path(
    "../knowledge_md/VMAX-S"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================================
# text cleaning
# =====================================================

def clean_text(text: str):

    if not text:
        return ""


    # PDF-specific whitespace

    text = text.replace(
        "\u00a0",
        " "
    )

    text = text.replace(
        "\u3000",
        " "
    )


    # runs of spaces

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )


    return text.strip()



# =====================================================
# Element -> Markdown
# =====================================================

def elements_to_markdown(elements):

    md = []


    for element in elements:


        text = clean_text(
            element.text
        )


        if not text:

            continue



        # --------------------------
        # heading
        # --------------------------

        if isinstance(
            element,
            Title
        ):


            md.append(
                "## " + text
            )

            md.append("")



        # --------------------------
        # list
        # --------------------------

        elif isinstance(
            element,
            ListItem
        ):


            md.append(
                "- " + text
            )



        # --------------------------
        # table
        # fast mode usually does not emit complex tables
        # --------------------------

        elif isinstance(
            element,
            Table
        ):


            md.append(
                text
            )

            md.append("")



        # --------------------------
        # body text
        # --------------------------

        elif isinstance(
            element,
            NarrativeText
        ):


            md.append(
                text
            )

            md.append("")



        # --------------------------
        # header / footer
        # --------------------------

        elif isinstance(
            element,
            (
                Header,
                Footer
            )
        ):

            continue



        # --------------------------
        # page break
        # --------------------------

        elif isinstance(
            element,
            PageBreak
        ):


            md.append(
                "\n---\n"
            )



        else:


            md.append(
                text
            )



    return "\n".join(md)



# =====================================================
# Markdown post-processing
# =====================================================

def clean_markdown(text):


    # drop consecutive blank lines

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )


    return text.strip()



# =====================================================
# a single PDF
# =====================================================

def convert_pdf(pdf_file: Path):
    print("=" * 60)
    print(
        "processing:",
        pdf_file.name
    )
    
    # ==============================
    # Unstructured fast mode
    # skip downloading the layout model
    # ==============================
    elements = partition_pdf(
        filename=str(pdf_file),

        # the key point:
        # do not call the YOLO layout model
        strategy="fast",
        # do not enable the layout table model
        infer_table_structure=False,
        # keep page breaks
        include_page_breaks=True,

        languages=[
            "chi_sim",
            "eng"
        ]

    )

    print(
        "element count:",
        len(elements)
    )

    markdown = elements_to_markdown(
        elements
    )

    markdown = clean_markdown(
        markdown
    )


    output_file = (
        OUTPUT_DIR /
        pdf_file.with_suffix(".md").name
    )

    output_file.write_text(
        markdown,
        encoding="utf-8"
    )

    print(
        "written:",
        output_file
    )



# =====================================================
# batch processing
# =====================================================

pdf_files = sorted(
    INPUT_DIR.glob(
        "*.pdf"
    )
)


print(
    f"PDFs found: {len(pdf_files)}"
)



for pdf in pdf_files:
    try:
        convert_pdf(
            pdf
        )
    except Exception as e:
        print(
            "failed:",
            pdf.name
        )
        print(
            repr(e)
        )

print(
    "\nall done"
)

## Tuned to this document family (iteratively refined)


In [ ]:
#!/usr/bin/env python
# coding: utf-8


import re
import fitz
from pathlib import Path



# =====================================================
# path configuration
# =====================================================

INPUT_DIR = Path(
    "../knowledge_path/VMAX-S"
)


OUTPUT_DIR = Path(
    "../knowledge_md/VMAX-S"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================================
# parameter configuration
# =====================================================

# drop the cover and table of contents if you do not need them
SKIP_PAGES = 0



# =====================================================
# header / footer
# =====================================================


HEADER_PATTERNS = [

    r".*\u4ea7\u54c1\u63cf\u8ff0.*",  # matches the Chinese "product description" title

]


FOOTER_PATTERNS = [

    r"SJ-\d+.*",

    r"\d{4}-\d{2}-\d{2}",

]


PAGE_NUMBER_PATTERN = re.compile(
    r"^\d+$"
)



# =====================================================
# basic cleaning
# =====================================================

def clean_text(text):


    replace_map = {

        "\u00a0": " ",
        "\u3000": " ",

        # PDF bullet markers
        "": "-",
        "": "-",
        "•": "-",
        "●": "-",
        "○": "-",
        "▪": "-",
        "▫": "-",
        "à": "-",

    }


    for k,v in replace_map.items():

        text=text.replace(
            k,
            v
        )


    text=re.sub(
        r"[ \t]+",
        " ",
        text
    )


    return text.strip()



# =====================================================
# header / footer detection
# =====================================================

def is_header(text):


    for p in HEADER_PATTERNS:

        if re.match(
            p,
            text
        ):

            return True


    return False



def is_footer(text):


    for p in FOOTER_PATTERNS:

        if re.search(
            p,
            text
        ):

            return True


    return False



# =====================================================
# heading detection
# =====================================================

def detect_heading(text):


    # level-3 heading
    if re.match(
        r"^\d+\.\d+\.\d+\s+.+",
        text
    ):

        return "### " + text



    # level-2 heading
    if re.match(
        r"^\d+\.\d+\s+.+",
        text
    ):

        return "## " + text



    # level-1 heading
    if re.match(
        r"^\d+\s+.+",
        text
    ):


        # guard against:
        # 2 U
        # 35 kg

        if re.match(
            r"^\d+\s+(U|kg|mm|cm|GB|TB|GHz)$",
            text,
            re.I
        ):

            return None


        return "# " + text



    return None



# =====================================================
# list handling
# =====================================================

def normalize_bullet(text):


    if text.startswith("-"):

        value=text[1:].strip()

        return "- " + value



    return text



# =====================================================
# decide whether to join wrapped lines
# =====================================================

def should_merge_line(
        previous,
        current,
        in_list=False
):


    if not previous:

        return False



    # heading
    if current.startswith("#"):

        return False



    # a new list
    if current.startswith("-"):

        return False



    # this line continues a list item

    if in_list:

        return True



    # the previous sentence is unfinished

    if not previous.endswith(
        (
            "\u3002",  # full-width CJK punctuation that can end a Chinese sentence
            "\uff01",
            "\uff1f",
            "\uff1b",
            "\uff1a",
            "\uff09",
            "\u3011",
            ".",
            ";",
            ":"
        )
    ):

        return True



    return False



# =====================================================
# table
# =====================================================

def extract_tables(page):


    tables=[]


    try:

        result=page.find_tables()


        for table in result.tables:

            tables.append(
                table.to_markdown()
            )


    except Exception:

        pass



    return tables



# =====================================================
# page parsing
# =====================================================

def parse_page(
        page,
        page_no
):


    markdown=[]


    markdown.append(
        f"<!-- page_number:{page_no} -->"
    )


    markdown.append("")



    # ---------------------
    # table
    # ---------------------

    for table in extract_tables(page):

        markdown.append(
            table
        )

        markdown.append("")



    # ---------------------
    # text extraction
    # ---------------------

    lines=[]


    blocks=page.get_text(
        "dict"
    )["blocks"]



    for block in blocks:


        if "lines" not in block:

            continue



        for line in block["lines"]:


            text="".join(
                span["text"]
                for span in line["spans"]
            )


            text=clean_text(
                text
            )


            if not text:

                continue



            y=line["bbox"][1]



            # header
            if y < 60:

                continue



            # footer
            if y > page.rect.height-60:

                continue



            if is_header(text):

                continue



            if is_footer(text):

                continue



            if PAGE_NUMBER_PATTERN.match(text):

                continue



            lines.append(
                text
            )



    # =================================================
    # text reassembly
    # =================================================

    result=[]


    buffer=""

    in_list=False


    pending_bullet=False



    for line in lines:



        # a lone bullet marker

        if line=="-":

            pending_bullet=True

            continue



        if pending_bullet:

            line="- "+line

            pending_bullet=False



        # heading

        heading=detect_heading(
            line
        )


        if heading:


            if buffer:

                result.append(
                    buffer
                )

                buffer=""


            result.append(
                heading
            )


            result.append("")

            in_list=False

            continue



        # list

        line=normalize_bullet(
            line
        )



        if line.startswith("-"):


            if buffer:

                result.append(
                    buffer
                )

                buffer=""


            result.append(
                line
            )


            in_list=True


            continue



        # ordinary text

        if should_merge_line(
            buffer,
            line,
            in_list
        ):

            buffer += line


        else:


            if buffer:

                result.append(
                    buffer
                )


            buffer=line


        in_list=False



    if buffer:

        result.append(
            buffer
        )



    return "\n".join(
        markdown + result
    )



# =====================================================
# Markdown tidy-up
# =====================================================

def clean_markdown(text):


    # excess blank lines

    text=re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )


    return text.strip()



# =====================================================
# PDF conversion
# =====================================================

def pdf_to_markdown(pdf_file):


    print(
        "processing:",
        pdf_file.name
    )


    pdf=fitz.open(
        pdf_file
    )


    pages=[]



    for page_no,page in enumerate(
        pdf,
        start=1
    ):


        if page_no <= SKIP_PAGES:

            continue



        pages.append(
            parse_page(
                page,
                page_no
            )
        )



    markdown="\n\n".join(
        pages
    )


    markdown=clean_markdown(
        markdown
    )



    output=(
        OUTPUT_DIR /
        pdf_file.with_suffix(".md").name
    )



    with open(
        output,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            markdown
        )


    print(
        "written:",
        output
    )



# =====================================================
# batch processing
# =====================================================

pdf_files=sorted(
    INPUT_DIR.glob(
        "*.pdf"
    )
)


print(
    "PDF count:",
    len(pdf_files)
)



for pdf in pdf_files:


    try:

        pdf_to_markdown(
            pdf
        )


    except Exception as e:

        print(
            "failed:",
            pdf.name,
            e
        )



print(
    "all done"
)